# 04 — Analysis

สังเคราะห์ผลจาก [`01_concept.ipynb`](01_concept.ipynb), [`02_from_scratch.ipynb`](02_from_scratch.ipynb),
[`03_experiment.ipynb`](03_experiment.ipynb) เข้าด้วยกัน แล้วตอบคำถามเปิดที่ตอบได้จริงจากสิ่งที่ทำมา

## 1. Results — สรุปผลจากทั้ง 3 โน้ตบุ๊ก

| แหล่งที่มา | สิ่งที่ทดสอบ | ผลลัพธ์ |
|---|---|---|
| `01_concept.ipynb` §4.4 | `Var(q·k)` โตตาม `d_k` หรือไม่ (pure Python, `TRIALS=2000`) | ใกล้เคียงทฤษฎีทุกค่า `d_k` (เช่น `d_k=512` → วัดได้ `506.56`) |
| `01_concept.ipynb` §5 | Attention output ถูกต้องตามสมการหรือไม่ (คำนวณด้วยมือ) | weight row 0 = `[0.506, 0.186, 0.307]`, sum=1 ทุกแถว |
| `02_from_scratch.ipynb` §4.1 | numpy implementation ตรงกับตัวเลขที่คำนวณด้วยมือหรือไม่ | ตรงกันภายใน `atol=1e-3` (`assert` ผ่าน) |
| `02_from_scratch.ipynb` §5 | output shape คงที่ `[n, d_model]` ไม่ว่า `h` เท่าไหร่หรือไม่ | ผ่านทุกค่า `h ∈ {1,2,4,8}` |
| `02_from_scratch.ipynb` §9 | `shared/src/attention` (extracted) ตรงกับ inline version หรือไม่ | ตรงกันทุกฟังก์ชัน (`assert` ผ่าน) |
| `03_experiment.ipynb` §3 | `shared/src/attention` ตรงกับ `torch.nn.functional.scaled_dot_product_attention` หรือไม่ | max diff `~8.1×10⁻⁸` ถึง `4.8×10⁻⁷` ทุก `d_k` (float32 epsilon) |
| `03_experiment.ipynb` §5 | `Var(q·k)` โตตาม `d_k` หรือไม่ (numpy vectorized, `TRIALS=20000`) | `d_k=512` วัดได้ `515.95` เทียบทฤษฎี `512` (<1% error) |

**สรุป:** hypothesis ทั้งหมดที่ตั้งไว้ใน `01_concept.ipynb` §3 ได้รับการยืนยันด้วยหลักฐานที่สอดคล้องกัน
3 ชั้น (คำนวณด้วยมือ → numpy → เทียบ torch) ไม่มีจุดใดขัดแย้งกัน

## 2. Visualization — Self-Attention vs. Recurrent vs. Convolutional Complexity

ยังไม่มีโน้ตบุ๊กไหนในแล็บนี้ visualize Table 1 (`../docs/paper-notes.md` หัวข้อ 8) เป็นตัวเลขจริง —
ที่ผ่านมามีแต่สูตร Big-O (`O(n²·d)`, `O(n·d²)`, `O(k·n·d²)`) ส่วนนี้แปลงสูตรเป็นกราฟจริงที่ `d=512`
(ค่า `d_model` ของ base model) เพื่อตอบคำถามเปิดข้อ 4 ใน [`../docs/open-questions.md`](../docs/open-questions.md)
โดยตรง: **crossover point อยู่ตรงไหน?**

In [ ]:
# P1 - Analytical (not measured) complexity comparison from Table 1's Big-O formulas.
import matplotlib.pyplot as plt
import numpy as np

d = 512  # d_model of the base model (see ../docs/paper-notes.md section 9)
k = 3  # typical convolution kernel width
n = np.logspace(0, 12, num=200, base=2)  # sequence length, 1 .. 4096

self_attention_ops = n**2 * d  # O(n^2 * d)
recurrent_ops = n * d**2  # O(n * d^2)
convolutional_ops = k * n * d**2  # O(k * n * d^2)

crossover_n = d  # n^2*d = n*d^2  =>  n = d

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(n, self_attention_ops, label="self-attention: O(n²·d)", color="#4C72B0")
ax.plot(n, recurrent_ops, label="recurrent: O(n·d²)", color="#DD8452")
ax.plot(n, convolutional_ops, label=f"convolutional: O(k·n·d²), k={k}", color="#55A868")
ax.axvline(crossover_n, color="gray", linestyle=":", label=f"crossover: n = d = {crossover_n}")

ax.set_xscale("log", base=2)
ax.set_yscale("log", base=2)
ax.set_xlabel("sequence length n")
ax.set_ylabel("operations per layer (relative units)")
ax.set_title(f"Per-layer complexity vs. sequence length (d_model={d}, fixed)")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"Crossover point: n = d = {crossover_n}")
print(f"At n=128 (n<d):  self-attention/recurrent ratio = {(128**2 * d) / (128 * d**2):.3f}")
print(f"At n=2048 (n>d): self-attention/recurrent ratio = {(2048**2 * d) / (2048 * d**2):.3f}")

**อ่านกราฟและตอบคำถามเปิดข้อ 4:** ตั้ง `n²d = nd²` แล้วแก้สมการ จะได้ **`n = d`** พอดี — ไม่ใช่แค่
"ประมาณ" แต่เป็นจุดตัดที่แม่นยำในเชิงพีชคณิต (ที่ `d_model=512` จุดตัดจึงอยู่ที่ `n=512` ไม่ใช่ค่าอื่น)
จากตัวเลขที่พิมพ์ไว้: ที่ `n=128` (`n < d`) self-attention ใช้ operation เพียง **0.25 เท่า** ของ recurrent
(ถูกกว่า) แต่ที่ `n=2048` (`n > d`) self-attention ใช้ operation **4 เท่า** ของ recurrent (แพงกว่า)

**ข้อควรระวัง (สำคัญ):** นี่คือ **การวิเคราะห์เชิงทฤษฎีจากสูตร Big-O เท่านั้น** ไม่ใช่การวัด latency/FLOPs
จริงบน hardware — คำถามเปิดข้อ 4 ถามถึง "ในทางปฏิบัติบน hardware ปัจจุบัน" ซึ่งกราฟนี้ **ยังตอบไม่ได้**
เพราะ recurrent layer มี sequential operation O(n) ที่ทำให้ **ขนานไม่ได้** บน GPU ในขณะที่ self-attention
มี sequential operation แค่ O(1) — แม้ operation count ดิบจะมากกว่าที่ `n>d` แต่ self-attention
อาจยังเร็วกว่าจริงบน GPU เพราะขนานได้เต็มที่ ในขณะที่ recurrent ต้องรอทีละ timestep คำถามนี้จึงถูก
**ตอบบางส่วน** (เชิงทฤษฎี/operation count) แต่ **ยังเปิดอยู่** ในแง่การวัดจริงบน hardware — ดูการอัปเดตใน
[`../docs/open-questions.md`](../docs/open-questions.md)

## 3. Interpretation

**Paper Finding vs. Interpretation (AGENTS.md #19):**

- *Paper Finding:* ผู้เขียนรายงานว่า self-attention มี complexity ต่อ layer เป็น `O(n²·d)` และ path
  length คงที่ `O(1)` เทียบกับ recurrent ที่เป็น `O(n·d²)` และ `O(n)` ตามลำดับ (ดู `../docs/paper-notes.md` §8)
- *Interpretation ของเรา (จาก §2 ด้านบน):* ข้อได้เปรียบด้าน operation count ของ self-attention
  ไม่ได้เป็นจริงเสมอไป — ขึ้นกับว่า `n` กับ `d` ตัวไหนมากกว่ากัน การที่ paper เน้นกรณี `n < d` (ประโยคสั้น
  เทียบกับ `d_model` ใหญ่) ไม่ใช่เรื่องบังเอิญ แต่เป็นเงื่อนไขที่ทำให้ argument ของ paper ใช้ได้ผลดีที่สุด
  เมื่อ `n` โตเกิน `d` (เช่น long-document, long-context) ข้อได้เปรียบด้าน raw operation count จะกลับด้าน
  — แม้ประโยชน์ด้าน parallelization (`O(1)` sequential ops) จะยังคงอยู่ก็ตาม

ทั้ง 3 hypothesis จาก `01_concept.ipynb` §3 ได้รับการยืนยันด้วยหลักฐาน 3 ชั้นที่สอดคล้องกัน
(ตัวเลขด้วยมือ → numpy → torch) ทำให้เชื่อมั่นได้ว่า mental model เรื่อง scaled dot-product attention
ที่สร้างไว้ตั้งแต่ `01_concept.ipynb` ถูกต้องทั้งในเชิงทฤษฎีและเชิง implementation

## 4. Limitations

ข้อจำกัดที่ผู้เขียน paper ระบุไว้เองอยู่ที่ [`../docs/limitations.md`](../docs/limitations.md)
ส่วนนี้คือข้อจำกัดของการทดลองใน **lab นี้เอง** (สะสมจากทั้ง 4 โน้ตบุ๊ก):

- ทุกการทดลองใช้ scale เล็กมาก (toy example `n≤8`, `d_k≤512`) เทียบกับโมเดลจริงที่ `n` มักหลักพัน-หมื่น
- `multi_head_attention` ใน `02_from_scratch.ipynb`/`shared/src` ใช้ **random projection ที่ไม่ train**
  ทุกข้อสรุปเรื่อง "head ต่างกัน" จึงเป็นแค่หลักฐานเชิงโครงสร้าง ไม่ใช่พฤติกรรมของโมเดลที่ train แล้วจริง
- `03_experiment.ipynb` ทดสอบเทียบกับ `torch` เฉพาะ scaled dot-product attention เดี่ยว ๆ, เฉพาะ
  `float32`, เฉพาะ CPU, ไม่มี mask — ยังไม่ครอบคลุม production configuration จริง (mixed precision, GPU,
  causal mask, multi-head เต็มรูปแบบ)
- กราฟ complexity ในหัวข้อ 2 เป็นการวิเคราะห์ทฤษฎี ไม่ใช่ benchmark วัดจริง (ดูคำเตือนในหัวข้อ 2)
- ทั้ง lab นี้ยังไม่มี encoder/decoder เต็มรูปแบบหรือ training loop จริง — เป็นการตรวจสอบ building block
  เท่านั้น ยังไม่ใช่การ reproduce ผล BLEU score ที่ paper รายงาน

## 5. Engineering Implications

ดู [`../docs/engineering-notes.md`](../docs/engineering-notes.md) สำหรับบริบทเต็ม ส่วนนี้เพิ่มข้อสังเกต
ใหม่ที่ได้จากการทดลองจริงในแล็บนี้:

- **จุดตัด `n=d` ในหัวข้อ 2 มีนัยเชิงวิศวกรรมโดยตรง**: เมื่อออกแบบระบบที่ต้องรองรับทั้ง short-sequence
  (`n<d_model`, เช่น sentence-level translation ตามที่ paper ทดลอง) และ long-context (`n>d_model`,
  เช่น document-level หรือ code understanding) การเลือกสถาปัตยกรรมอาจไม่ใช่คำตอบเดียวที่เหมาะกับทุกกรณี
  — นี่คือแรงจูงใจส่วนหนึ่งของงานยุคหลังที่ผสม local/global attention (เช่น Longformer, BigBird)
- **การ verify เทียบ library implementation** (`03_experiment.ipynb`) ที่ error อยู่ระดับ float32
  epsilon คงที่ทุก `d_k` (ไม่โตขึ้นตาม `d_k`) เป็นสัญญาณที่ดีว่า implementation ไม่มี numerical
  instability ซ่อนอยู่ — ถ้า error โตขึ้นตาม `d_k` แทน จะเป็นสัญญาณเตือนของปัญหา accumulation error
  ที่ควรตรวจสอบก่อนนำไป train จริง (โดยเฉพาะที่ precision ต่ำกว่า float32)
- **การแยก `shared/src/attention` ออกจาก notebook** (§9 ใน `02_from_scratch.ipynb`) ไม่ใช่แค่เรื่อง
  organization — มันทำให้ `03_experiment.ipynb` import โค้ดเดียวกับที่ `02` verify ไว้แล้วได้โดยตรง
  ลดความเสี่ยงที่จะเกิด "โค้ดสองชุดที่ควรจะเหมือนกันแต่ค่อย ๆ เพี้ยนออกจากกัน" (implementation drift)
  ระหว่าง notebook ในระยะยาว

## 6. Open Questions

ดู [`../docs/open-questions.md`](../docs/open-questions.md) สำหรับรายการเต็ม สถานะหลังจบ lab นี้:

**ตอบได้บางส่วนแล้ว:**

- คำถามข้อ 4 (crossover point) — ตอบได้เชิงทฤษฎี/operation count: `n = d` พอดี (ดูหัวข้อ 2 ด้านบน)
  แต่ยัง **ไม่ได้วัดจริงบน hardware** (ยังต้องพึ่ง benchmark จริง ซึ่งเชื่อมกับคำถามข้อ 5 และ lab
  `18_flashattention`/`19_flashattention2` ในอนาคต)

**ยังเปิดอยู่ทั้งหมด (ตามเดิม):**

- คำถามข้อ 1–3 (variance ที่ `d_k` ต่ำมาก, สิ่งที่แต่ละ head เรียนรู้จริง, sinusoidal vs learned PE
  ตอน extrapolate) — ต้องมี **trained model จริง** ถึงจะตอบได้ ซึ่งเกินขอบเขตของ lab นี้ (ดู Limitations)
- คำถามข้อ 5–7 (benchmark จริงตาม context length, KV cache/MQA/GQA, pre-norm vs post-norm)
  — รอ lab ในอนาคตตามที่ระบุไว้ในคำถามแต่ละข้อ

## สถานะของ Lab 01

Core numerical/conceptual investigation ของ scaled dot-product attention, multi-head attention,
และ positional encoding **เสร็จสมบูรณ์**: concept → implementation → verification → analysis
ครบทั้ง pipeline ตาม `AGENTS.md` §32 (Definition of Done) ยกเว้นสองข้อที่ยังไม่ทำ (ตั้งใจ อยู่นอกขอบเขต
ของ lab นี้): (1) reproduce ผล BLEU จาก training จริง, (2) encoder/decoder เต็มรูปแบบ — ทั้งสองข้อ
รอ lab ถัดไปตามที่ระบุไว้ใน [`../README.md`](../README.md)

## Next Notebook

→ [`05_training_concepts.ipynb`](05_training_concepts.ipynb): รายละเอียดฝั่ง training ที่ paper ระบุไว้
และเป็นตัวตัดสินว่าโมเดลจะเทรนสำเร็จหรือพัง — learning rate warmup, label smoothing,
embedding × `√d_model`, ตำแหน่งที่ใส่ dropout (อธิบายและสาธิตเชิงตัวเลข โดยไม่ต้อง train จริง)